# Asset Path Audit and Correction
This notebook scans HTML/CSS/JavaScript files in the project for image references, verifies whether the referenced files exist, and corrects any broken relative paths to the `assets/` folder.

In [ ]:
import os
import pathlib
import re
import json

In [ ]:
root = pathlib.Path('c:/Users/r u s t y/OneDrive/Desktop/synthesis-rusty-spring-2026')
assets_dir = root / 'assets'
html_dir = root / 'htmls'
allowed_extensions = {'.html', '.css', '.js'}
print('Root:', root)
print('Assets directory exists:', assets_dir.exists())
print('HTML directory exists:', html_dir.exists())

In [ ]:
ref_pattern = re.compile(r'(?:src|href)\s*=\s*["]([^"]+)["]|url\(\s*["]?([^")]+)["]?\s*\)')
image_refs = []
for path in root.rglob('*'):
    if path.suffix.lower() in allowed_extensions:
        text = path.read_text(encoding='utf-8', errors='ignore')
        for match in ref_pattern.finditer(text):
            ref = match.group(1) or match.group(2)
            if ref and not ref.startswith(('http:', 'https:', 'data:', 'mailto:', '#')):
                image_refs.append((path, ref))
print('Collected references:', len(image_refs))

In [ ]:
missing = []
suggestions = []
for path, ref in image_refs:
    if ref.startswith('/'):
        candidate = root / ref.lstrip('/')
    else:
        candidate = (path.parent / ref).resolve()
    if not candidate.exists():
        missing.append((path, ref, candidate))
        if ref.startswith('assets/') and path.parent == html_dir:
            suggestions.append((path, ref, '../' + ref))
        elif ref.startswith('/assets/'):
            suggestions.append((path, ref, '../' + ref.lstrip('/')))
        elif ref.startswith('assets/') and path.parent != html_dir:
            suggestions.append((path, ref, ref))
print('Missing references:', len(missing))
for path, ref, candidate in missing[:20]:
    print(path, '->', ref, 'expected', candidate)

In [ ]:
def correct_path(path, ref):
    if ref.startswith('/assets/'):
        return '../' + ref.lstrip('/')
    if path.parent == html_dir and ref.startswith('assets/'):
        return '../' + ref
    return ref

update_log = []
for path, ref in image_refs:
    corrected = correct_path(path, ref)
    if corrected != ref:
        text = path.read_text(encoding='utf-8', errors='ignore')
        new_text = text.replace(ref, corrected)
        if new_text != text:
            path.write_text(new_text, encoding='utf-8')
            update_log.append((path, ref, corrected))
print('Updated references:', len(update_log))
for record in update_log[:50]:
    print(record)